Exploratory paired comparison on the previously evaluated CShaper windows. Joint event selection is inspired by Schiegg et al. https://doi.org/10.1093/bioinformatics/btu603. Scores are fixed geometric heuristics.

In [ ]:

import sys, subprocess, json, re, hashlib
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel','beautifulsoup4'],check=True)
import nibabel, requests
from bs4 import BeautifulSoup
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
def normalize(a):
    lo, hi = np.percentile(a, (1, 99.8))
    return np.clip((a.astype(np.float32)-lo)/max(1,hi-lo),0,1)
def block(a,b):
    return nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU(),
                         nn.Conv3d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU())
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc, self.deep, self.dec, self.head = block(1,8), block(8,16), block(24,8), nn.Conv3d(8,1,1)
    def forward(self,x):
        a=self.enc(x)
        b=self.deep(F.avg_pool3d(a,(1,2,2)))
        return self.head(self.dec(torch.cat([a,F.interpolate(b,size=a.shape[2:],mode='trilinear',align_corners=False)],1)))


assert torch.cuda.is_available(), 'Kaggle GPU required'
torch.manual_seed(42)
rng=np.random.default_rng(42)
out=Path('/kaggle/working')
cache=Path('/tmp/cshaper')
cache.mkdir(exist_ok=True)
train=[f'Sample{i:02d}' for i in range(4,18)]
test=['Sample18','Sample19','Sample20']

def listing(folder):
    response=requests.get('https://drive.google.com/embeddedfolderview',params={'id':folder},timeout=40)
    response.raise_for_status()
    entries={}
    for anchor in BeautifulSoup(response.text,'html.parser').select('a'):
        title=anchor.select_one('.flip-entry-title')
        match=re.search(r'/(?:folders|d)/([\w-]+)',anchor.get('href',''))
        if title and match:
            entries[title.get_text(strip=True)]=match[1]
    return entries

def frames(entries, sample, suffix):
    pattern=rf'{sample}_(\d+)_{suffix}\.nii\.gz'
    return {int(m[1]): key for name,key in entries.items() if (m:=re.fullmatch(pattern,name))}

def fetch(key):
    path=cache/f'{key}.nii.gz'
    if not path.exists():
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry
        with requests.Session() as session:
            session.mount('https://', HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2,
                status_forcelist=(429, 500, 502, 503, 504))))
            with session.get('https://drive.usercontent.google.com/download',
                             params=dict(id=key, export='download', confirm='t'),
                             stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                chunks = response.iter_content(1024 * 1024)
                head = next(chunks, b'')
                if not head.startswith(bytes((31, 139))):
                    raise ValueError(f'Drive did not serve a gzip file for {key}')
                staging = path.with_suffix('.part')
                with staging.open('wb') as output:
                    output.write(head)
                    for chunk in chunks:
                        output.write(chunk)
                staging.replace(path)
    return nibabel.load(path), hashlib.sha256(path.read_bytes()).hexdigest()


import base64, io, csv
PAIRS = {'Sample18': {1: ['b41d59a7872b015d4cee2c3848a8595f0d8eb3f78b70e6051ad3429768147fe1', '18430e4459485a7c2e2027e48a318f95950f19c51dc6085e0cf1020a98718656'], 80: ['1e0dea96a391905d6886617de86cb2ce5f9b6a8cabe51558c0a33eb49b1d6a13', '30639b8c85eabffd688b4e63486caef65dd3127fdb6610692d95573932809211'], 159: ['9414544bc04d2fda8a939997e698a53b9ff6851710c1a385bab4855b971598c4', 'cfc9649b550525aae285975d4fb4160e4ef089d661b309770e9d5d34ba0dce46']}, 'Sample19': {1: ['d0f74454f7451c0103cf8d498aa630c5a3bdeda8642bdd492af5f88108cb91a8', '7334900292855f0f86b35784cedaf373a1e5fbf47afa1189064e643ff81c97be'], 80: ['5c6eaab34be286c41c41439169a335d05d61bc02b027683e50ef370315f0e90c', 'b5d4a3b5860d8cefdd056c2a9682eacbcc6bb6d103bb9f7868ac16d52b72387b'], 159: ['5d5fb285687a8f4fc8c2625a7f8e548e578b7a1c7f196712e71b74a3d441de5f', 'f0c1b48d1b3760e116019a20e9707f310ff9245934d1fbb297d3db15480147f7']}, 'Sample20': {1: ['c050646e68af05c42c14147fce4a419e2ba0ecb19a5e6057adb4f019ed84e029', '3a501e8169267c2a127c536a6b43ff957efacaa54c07de080404dad8c7f2b95c'], 85: ['2a64dea324f0a0822a77c1d3619daffa4edb0afa357612dcee8b8c00fa13e299', 'dcd79de8d42a33600a91777a1fe72cdd675c6cb5e51229798f6d27d25ac1df49'], 169: ['b9defba950a05531babcaa5869c793ec420f0eeaf8e3e77cbbf71df4e0485ebe', '640b038bfc4dba6dfe002658b0157a96be900b8d4af324d7185800bc2d9750e6']}}
WEIGHT = '...'
WEIGHT_SHA = '405fe6f4f6682038a54817ba75d860fd868ff39f9de00fde054fddb52a1089a4'

weights=base64.b64decode(WEIGHT)
assert hashlib.sha256(weights).hexdigest()==WEIGHT_SHA
net=Detector().cuda()
net.load_state_dict(torch.load(io.BytesIO(weights), map_location='cpu', weights_only=True)['model'])
net.eval()
response=requests.get('https://drive.usercontent.google.com/download',
                      params=dict(id='1zBqmyu9kZJ9NMM1YjHzQrIGftd_Q4BJk',export='download',confirm='t'),timeout=60)
response.raise_for_status()
names={int(k):v for k,v in list(csv.reader(io.StringIO(response.text)))[1:] if k.isdigit()}
folders=listing('151pBI7d_JfkOz4eQI4R4F2u_cKtlftBi')

def assignments(a,b,gate):
    d=cdist(a,b)
    cost=np.where(d<=gate,d,(min(d.shape)+1)*(gate+1))
    i,j=linear_sum_assignment(cost)
    return [(int(u),int(v)) for u,v in zip(i,j) if d[u,v]<=gate]

@torch.inference_mode()
def observations(raw_key,mask_key,expected_mask):
    raw,raw_hash=fetch(raw_key)
    mask,mask_hash=fetch(mask_key)
    if expected_mask is not None:
        assert mask_hash==expected_mask
    assert raw.shape==mask.shape and np.allclose(raw.affine,mask.affine,atol=1e-4)
    image=normalize(np.asarray(raw.dataobj).transpose(2,1,0))
    labels=np.asarray(mask.dataobj).transpose(2,1,0)
    ids=np.unique(labels[labels>0])
    assert all(int(k) in names for k in ids)
    truth=np.asarray(ndi.center_of_mass(labels>0,labels,ids)).reshape(-1,3)
    scores=np.empty_like(image,dtype=np.float32)
    for start in range(0,len(image),8):
        lo,hi=max(0,start-4),min(len(image),start+12)
        end=min(len(image),start+8)
        x=torch.as_tensor(image[lo:hi],dtype=torch.float32,device='cuda')[None,None]
        scores[start:end]=net(x).sigmoid()[0,0,start-lo:end-lo].cpu().numpy()
    pred=peak_local_max(scores,threshold_abs=0.5,min_distance=2,exclude_border=False)
    matched={i:names[int(ids[j])] for i,j in assignments(pred,truth,3)}
    return pred,matched,{names[int(k)] for k in ids},raw_hash


from collections import defaultdict

# Canonical binary suffixes plus explicitly named early branches.
early = {
    'P0': ('AB','P1'), 'P1': ('EMS','P2'), 'EMS': ('MS','E'),
    'P2': ('C','P3'), 'P3': ('D','P4'), 'P4': ('Z2','Z3'),
}

def reference_divisions(before, after):
    disappeared = before-after
    new = after-before
    events = set()
    for parent in disappeared:
        options = ([early[parent]] if parent in early else
                   [(parent+x,parent+y) for x,y in ('ap','lr','dv')])
        found = [tuple(sorted(pair)) for pair in options if set(pair) <= new]
        if len(found) == 1:
            events.add((parent,*found[0]))
    return events

def split_candidates(a,b,links,gate):
    occupied = {j for _,j in links}
    anchor = dict(links)
    candidates = []
    across = cdist(a,b)
    for i,j in anchor.items():
        for k in np.flatnonzero(across[i] <= gate):
            if k in occupied:
                continue
            sister = np.linalg.norm(b[j]-b[k])
            midpoint = np.linalg.norm(a[i]-(b[j]+b[k])/2)
            if 2 <= sister <= gate and midpoint <= gate/2:
                candidates.append((midpoint,i,j,int(k)))
    used_parents = set()
    splits = []
    for _,i,j,k in sorted(candidates):
        if i not in used_parents and k not in occupied:
            occupied.add(k)
            used_parents.add(i)
            splits.append((i,j,k))
    return splits


from itertools import combinations
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix

def joint_events(a, b, gate):
    distances = cdist(a,b)
    events, scores = [], []
    for i in range(len(a)):
        nearby = np.flatnonzero(distances[i] <= gate)
        for j in nearby:
            events.append((i,int(j)))
            scores.append(1 - distances[i,j]/gate)
        for j,k in combinations(nearby,2):
            separation = np.linalg.norm(b[j]-b[k])
            midpoint = np.linalg.norm(a[i]-(b[j]+b[k])/2)
            if not (2 <= separation <= 2*gate and midpoint <= gate/2):
                continue
            imbalance = abs(distances[i,j]-distances[i,k])
            events.append((i,int(j),int(k)))
            scores.append(1.35 - 1.6*midpoint/gate - 0.5*imbalance/gate
                          - 0.35*separation/gate)
    if not events:
        return [], [], []
    rows, cols = [], []
    for column,event in enumerate(events):
        for row in (event[0], *(len(a)+j for j in event[1:])):
            rows.append(row)
            cols.append(column)
    incidence = coo_matrix((np.ones(len(rows)),(rows,cols)),shape=(len(a)+len(b),len(events))).tocsc()
    solution = milp(-np.asarray(scores),integrality=np.ones(len(events)),
                    bounds=Bounds(0,1),constraints=LinearConstraint(incidence,0,1),
                    options={'time_limit':30})
    if not solution.success:
        raise RuntimeError(f'Joint assignment failed: {solution.message}')
    selected = [event for event,x in zip(events,solution.x) if x > 0.5]
    return ([e for e in selected if len(e)==2],[e for e in selected if len(e)==3],
            [e for e in events if len(e)==3])

rows=[]
windows={}
report=out/'joint_link_summary.json'
for sample in PAIRS:
    children=listing(folders[sample])
    images=frames(listing(children['RawNuc']),sample,'rawNuc')
    masks=frames(listing(children['SegNuc']),sample,'segNuc')
    available=set(images)&set(masks)
    audited={t+i:h for t,hashes in PAIRS[sample].items() for i,h in enumerate(hashes)}
    # Evaluate three fixed 10-frame windows around early, middle and late stages.
    chosen=[]
    for anchor in PAIRS[sample]:
        starts=[t for t in sorted(available) if set(range(t,t+10))<=available
                and not set(range(t,t+10)).intersection(chosen)]
        assert starts, f'No disjoint consecutive window for {sample}'
        start=min(starts,key=lambda t:(abs(t-anchor),t))
        chosen.extend(range(start,start+10))
    windows[sample]=[chosen[i:i+10] for i in range(0,len(chosen),10)]
    for window in windows[sample]:
        previous=None
        for t in window:
            current=observations(images[t],masks[t],audited.get(t))
            if previous is not None:
                a,ma,na,ha=previous
                b,mb,nb,hb=current
                divisions=reference_divisions(na,nb)
                detectable=set(ma.values()) & set(mb.values())
                detectable_splits=sum(p in ma.values() and x in mb.values() and y in mb.values()
                                      for p,x,y in divisions)
                gate=9
                base=assignments(a,b,gate)
                baseline_splits=split_candidates(a,b,base,gate)
                joint_links,joint_splits,candidates=joint_events(a,b,gate)

                def name_splits(events):
                    return {(ma[i],*sorted((mb[j],mb[k]))) for i,j,k in events
                            if i in ma and j in mb and k in mb}

                for method,links,splits in (('baseline',base,baseline_splits),
                                             ('joint',joint_links,joint_splits)):
                    known=[(ma[i],mb[j]) for i,j in links if i in ma and j in mb]
                    named_splits=name_splits(splits)
                    rows.append(dict(sample=sample,t=t-1,method=method,reference=len(na&nb),
                        both_detected=len(detectable),predicted_links=len(links),
                        matched_endpoints_links=len(known),correct=sum(x==y for x,y in known),
                        reference_splits=len(divisions),detectable_splits=detectable_splits,
                        candidate_splits_recovered=len(name_splits(candidates)&divisions) if method=='joint' else None,
                        predicted_splits=len(splits),matched_endpoint_splits=len(named_splits),
                        correct_splits=len(named_splits&divisions),raw_sha256=[ha,hb]))
            previous=current
    report.write_text(json.dumps(dict(weight_sha256=WEIGHT_SHA,windows=windows,rows=rows,
        division_reference='Canonical name rules and adjacent parent disappearance plus both daughter appearances. Partial reference.',
        scope='Adjacent links within consecutive windows'),indent=2))
    print(sample,len(rows),flush=True)
assert len(rows)==162
